# Visual 3: Income and Wage Distribution Comparison (Austin TX and US)

This visualization highlights income inequality and the disconnect between rising home values and stagnant wages. It compares the distribution of household incomes in Austin with national averages, alongside a wage distribution comparison for Texas versus the U.S.

In [ ]:
# Import required libraries
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

In [ ]:
# Load the datasets
income_df = pd.read_csv('../data/austin_household_income.csv')
wage_df = pd.read_csv('../data/texas_wage_distribution.csv')

# Preview the data
print("Income Data Shape:", income_df.shape)
print("\nIncome Data Columns:", income_df.columns.tolist())
print("\nWage Data Shape:", wage_df.shape)
print("\nWage Data Columns:", wage_df.columns.tolist())

In [ ]:
# Prepare income data - filter for Austin and US
income_austin = income_df[income_df['Place'] == 'Austin, TX'].copy()
income_us = income_df[income_df['Place'] == 'United States'].copy()

# Prepare wage data - filter for Texas and US
wage_texas = wage_df[wage_df['State'] == 'Texas'].copy()
wage_us = wage_df[wage_df['State'] == 'United States'].copy()

# Check available years
print("Income data years:", sorted(income_austin['Year'].unique()))
print("Wage data years:", sorted(wage_texas['Year'].unique()))

In [ ]:
import plotly.graph_objects as go

# Create an interactive version with year slider (Income Distribution Only)
def create_interactive_comparison_with_slider():
    """
    Create an interactive visualization with a year slider showing household income distribution
    """
    
    # Get all available years for income data
    years = sorted(income_austin['Year'].unique())
    
    # Create initial figure for first year
    initial_year = years[-1]
    
    # Initialize the figure (single plot, no subplots)
    fig = go.Figure()
    
    # Define colors
    color_austin_texas = '#FF6B35'  # Warm orange
    color_us = '#4A90B5'  # Cool gray-blue
    
    # Austin population data by year (approximate values)
    population_data = {
        2012: 842592,
        2013: 869809,
        2014: 903543,
        2015: 932388,
        2016: 955830,
        2017: 971844,
        2018: 990899,
        2019: 1013901,
        2020: 1037878,
        2021: 1056844,
        2022: 1068370,
        2023: 1028225
    }
    
    # Create frames for animation/slider
    frames = []

    def filtered(df, year, id_col):
        return df[df['Year'] == year].sort_values(id_col)
    
    def get_dynamic_annotations(year, austin_data, us_data):
        """Generate year-specific annotations with insights"""
        annotations = []
        
        # Population annotation (always shown)
        pop = population_data.get(year, 0)
        annotations.append(dict(
            text=f"<b>Austin Population:</b><br>{pop:,}",
            xref="paper", yref="paper",
            x=0.02, y=0.98,
            showarrow=False,
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor=color_austin_texas,
            borderwidth=2,
            borderpad=8,
            font=dict(size=12, color="#333333"),
            align="left",
            xanchor="left",
            yanchor="top"
        ))
        
        # Calculate key metrics
        austin_high_income = austin_data[austin_data['Household Income Bucket ID'] >= 14]['share'].sum() * 100
        us_high_income = us_data[us_data['Household Income Bucket ID'] >= 14]['share'].sum() * 100
        income_gap = austin_high_income - us_high_income
        
        # High income concentration annotation
        annotations.append(dict(
            text=f"<b>High Income ($150k+):</b><br>Austin: {austin_high_income:.1f}%<br>U.S.: {us_high_income:.1f}%<br>Gap: +{income_gap:.1f}%",
            xref="paper", yref="paper",
            x=0.02, y=0.85,
            showarrow=False,
            bgcolor="rgba(255,107,53,0.1)",
            bordercolor=color_austin_texas,
            borderwidth=2,
            borderpad=6,
            font=dict(size=10, color="#333333"),
            align="left",
            xanchor="left",
            yanchor="top"
        ))
        
        # Year-specific insights
        if year == 2020:
            annotations.append(dict(
                text="<b>COVID-19 Impact:</b><br>Pandemic shifts labor market",
                xref="x", yref="y",
                x=austin_data['Household Income Bucket'].iloc[5],
                y=austin_data['share'].iloc[5] * 100 + 2,
                showarrow=True,
                arrowhead=2,
                arrowsize=1,
                arrowwidth=2,
                arrowcolor="#E63946",
                ax=60, ay=-40,
                bordercolor="#E63946",
                borderwidth=2,
                borderpad=4,
                bgcolor="rgba(255,255,255,0.95)",
                font=dict(size=10, color="#333333")
            ))
        elif year == 2023:
            annotations.append(dict(
                text="<b>Austin's higher $200k+ share<br>reflects wealth concentration</b>",
                xref="x", yref="y",
                x=austin_data['Household Income Bucket'].iloc[-2],
                y=austin_data['share'].iloc[-2] * 100 + 1.2,
                showarrow=True,
                arrowhead=2,
                arrowsize=1,
                arrowwidth=2,
                arrowcolor=color_austin_texas,
                ax=80, ay=-40,
                bordercolor=color_austin_texas,
                borderwidth=2,
                borderpad=4,
                bgcolor="rgba(255,255,255,0.95)",
                font=dict(size=10, color="#333333")
            ))
        elif year >= 2021:
            # Middle class squeeze
            middle_class = austin_data[(austin_data['Household Income Bucket ID'] >= 7) & 
                                       (austin_data['Household Income Bucket ID'] <= 10)]['share'].sum() * 100
            annotations.append(dict(
                text=f"<b>Middle Class ($50-100k):</b><br>{middle_class:.1f}% of households",
                xref="x", yref="y",
                x=austin_data['Household Income Bucket'].iloc[8],
                y=austin_data['share'].iloc[8] * 100 + 1.5,
                showarrow=True,
                arrowhead=2,
                arrowsize=1,
                arrowwidth=2,
                arrowcolor="#2A9D8F",
                ax=-60, ay=-30,
                bordercolor="#2A9D8F",
                borderwidth=2,
                borderpad=4,
                bgcolor="rgba(255,255,255,0.95)",
                font=dict(size=10, color="#333333")
            ))
        elif year <= 2014:
            # Earlier years - growth context
            annotations.append(dict(
                text=f"<b>Pre-Tech Boom Era</b><br>More balanced distribution",
                xref="paper", yref="paper",
                x=0.5, y=0.5,
                showarrow=False,
                bgcolor="rgba(74,144,181,0.1)",
                bordercolor=color_us,
                borderwidth=2,
                borderpad=6,
                font=dict(size=11, color="#333333"),
                xanchor="center"
            ))
        
        return annotations

    for year in years:
        # Filter data for this year
        income_austin_year = filtered(income_austin, year, 'Household Income Bucket ID')
        income_us_year = filtered(income_us, year, 'Household Income Bucket ID')
        
        frame_data = [
            # Income - Austin
            go.Bar(
                x=income_austin_year['Household Income Bucket'],
                y=income_austin_year['share'] * 100,
                name='Austin, TX',
                marker_color=color_austin_texas,
                hovertemplate='<b>%{x}</b><br>Austin: %{y:.2f}%<br><extra></extra>',
                legendgroup='region1',
                showlegend=True
            ),
            # Income - US
            go.Bar(
                x=income_us_year['Household Income Bucket'],
                y=income_us_year['share'] * 100,
                name='United States',
                marker_color=color_us,
                hovertemplate='<b>%{x}</b><br>U.S.: %{y:.2f}%<br><extra></extra>',
                legendgroup='region2',
                showlegend=True
            )
        ]
        
        # Get dynamic annotations for this year
        frame_annotations = get_dynamic_annotations(year, income_austin_year, income_us_year)
        
        frames.append(go.Frame(
            data=frame_data,
            name=str(year),
            layout=go.Layout(
                title_text=f'Household Income Distribution Comparison ({year})<br>' +
                          '<sub>Comparing Austin with National Averages</sub>',
                annotations=frame_annotations
            )
        ))
    
    # Add initial data
    income_austin_init = filtered(income_austin, initial_year, 'Household Income Bucket ID')
    income_us_init = filtered(income_us, initial_year, 'Household Income Bucket ID')
    
    # Income panel
    fig.add_trace(
        go.Bar(
            x=income_austin_init['Household Income Bucket'],
            y=income_austin_init['share'] * 100,
            name='Austin, TX',
            marker_color=color_austin_texas,
            hovertemplate='<b>%{x}</b><br>Austin: %{y:.2f}%<br><extra></extra>',
            legendgroup='region1'
        )
    )
    
    fig.add_trace(
        go.Bar(
            x=income_us_init['Household Income Bucket'],
            y=income_us_init['share'] * 100,
            name='United States',
            marker_color=color_us,
            hovertemplate='<b>%{x}</b><br>U.S.: %{y:.2f}%<br><extra></extra>',
            legendgroup='region2'
        )
    )
    
    # Add frames
    fig.frames = frames
    
    # Update axes
    fig.update_xaxes(title_text="Income Bracket", tickangle=-45)
    fig.update_yaxes(title_text="Share of Households (%)", range=[0, 18])
    
    # Slider & buttons
    sliders = [dict(
        active=len(years) - 1,
        yanchor="top",
        y=-0.15,
        xanchor="left",
        x=0.1,
        currentvalue=dict(prefix="Year: ", visible=True, xanchor="center", font=dict(size=16)),
        pad=dict(b=10, t=10),
        len=0.8,
        transition=dict(duration=300),
        steps=[
            dict(
                args=[[str(year)], dict(frame=dict(duration=300, redraw=True), mode="immediate", transition=dict(duration=300))],
                method="animate",
                label=str(year)
            )
            for year in years
        ]
    )]
    
    # Layout
    fig.update_layout(
        title={
            'text': f'Household Income Distribution Comparison ({initial_year})<br><sub>Comparing Austin with National Averages</sub>',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20}
        },
        barmode='group',
        height=1300,
        width=1500,
        showlegend=True,
        legend=dict(
            title=dict(text='Region'),
            orientation='v',
            yanchor='top',
            y=0.98,
            xanchor='right',
            x=0.98,
            font=dict(size=12)
        ),
        sliders=sliders,
        font=dict(size=11),
        hovermode='x unified',
        paper_bgcolor='white',
        plot_bgcolor='rgba(240,240,240,0.5)',
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                x=0.5,
                y=-0.25,
                xanchor="center",
                yanchor="top",
                pad=dict(r=10, t=10),
                buttons=[
                    dict(
                        label="Play",
                        method="animate",
                        args=[None, dict(
                            frame=dict(duration=500, redraw=True),
                            fromcurrent=True,
                            mode="immediate",
                            transition=dict(duration=300)
                        )]
                    ),
                    dict(
                        label="Pause",
                        method="animate",
                        args=[[None], dict(
                            frame=dict(duration=0, redraw=False),
                            mode="immediate",
                            transition=dict(duration=0)
                        )]
                    )
                ]
            )
        ]
    )
    
    # Updated annotation - Get initial annotations
    initial_annotations = get_dynamic_annotations(initial_year, income_austin_init, income_us_init)
    
    # Add all initial annotations
    for annotation in initial_annotations:
        fig.add_annotation(annotation)
    
    return fig

# Create and show the interactive visualization with slider
fig_interactive = create_interactive_comparison_with_slider()
fig_interactive.show()


In [ ]:
import plotly.graph_objects as go

# Create an interactive version with year slider (Wage Distribution Only)
def create_interactive_wage_comparison_with_slider():
    """
    Create an interactive visualization with a year slider showing wage distribution
    """
    
    # Get all available years for wage data
    years = sorted(wage_texas['Year'].unique())
    
    # Create initial figure for first year
    initial_year = years[-1]
    
    # Initialize the figure (single plot, no subplots)
    fig = go.Figure()
    
    # Define colors
    color_texas = '#BF5700'  # Texas burnt orange
    color_us = '#4A90B5'  # Cool gray-blue
    
    # Texas population data by year (approximate values)
    population_data = {
        2012: 26448193,
        2013: 26956958,
        2014: 27469114,
        2015: 27862596,
        2016: 28304596,
        2017: 28701845,
        2018: 29087070,
        2019: 29472295,
        2020: 29145505,
        2021: 29558864,
        2022: 30029572,
        2023: 30503301
    }
    
    # Create frames for animation/slider
    frames = []

    def filtered_wage(df, year, id_col):
        return df[df['Year'] == year].sort_values(id_col)
    
    def get_dynamic_wage_annotations(year, texas_data, us_data):
        """Generate year-specific annotations with insights"""
        annotations = []
        
        # Population annotation (always shown) - left side
        pop = population_data.get(year, 0)
        annotations.append(dict(
            text=f"<b>Texas Population:</b><br>{pop:,}",
            xref="paper", yref="paper",
            x=0.02, y=0.98,
            showarrow=False,
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor=color_texas,
            borderwidth=2,
            borderpad=8,
            font=dict(size=14, color="#333333"),
            align="left",
            xanchor="left",
            yanchor="top"
        ))
        
        # Calculate key metrics
        texas_high_wage = texas_data[texas_data['Wage Bin ID'] >= 14]['share'].sum() * 100
        us_high_wage = us_data[us_data['Wage Bin ID'] >= 14]['share'].sum() * 100
        wage_gap = texas_high_wage - us_high_wage
        
        # High wage concentration annotation - MOVED TO RIGHT OF POPULATION
        annotations.append(dict(
            text=f"<b>High Wages ($150k+):</b><br>Texas: {texas_high_wage:.1f}%<br>U.S.: {us_high_wage:.1f}%<br>Gap: {wage_gap:+.1f}%",
            xref="paper", yref="paper",
            x=0.22, y=0.98,  # Moved to the right
            showarrow=False,
            bgcolor="rgba(191,87,0,0.1)",
            bordercolor=color_texas,
            borderwidth=2,
            borderpad=8,
            font=dict(size=14, color="#333333"),
            align="left",
            xanchor="left",
            yanchor="top"
        ))
        
        # Year-specific insights
        if year == 2020:
            annotations.append(dict(
                text="<b>COVID-19 Impact:</b><br>Pandemic affects wage distribution",
                xref="x", yref="y",
                x=texas_data['Wage Bin'].iloc[5],
                y=texas_data['share'].iloc[5] * 100 + 2,
                showarrow=True,
                arrowhead=2,
                arrowsize=1,
                arrowwidth=2,
                arrowcolor="#E63946",
                ax=60, ay=-40,
                bordercolor="#E63946",
                borderwidth=2,
                borderpad=4,
                bgcolor="rgba(255,255,255,0.95)",
                font=dict(size=12, color="#333333")
            ))
        elif year == 2023:
            annotations.append(dict(
                text="<b>Texas wage distribution<br>closely tracks national pattern</b>",
                xref="x", yref="y",
                x=texas_data['Wage Bin'].iloc[-2],
                y=texas_data['share'].iloc[-2] * 100 + 1.2,
                showarrow=True,
                arrowhead=2,
                arrowsize=1,
                arrowwidth=2,
                arrowcolor=color_texas,
                ax=80, ay=-40,
                bordercolor=color_texas,
                borderwidth=2,
                borderpad=4,
                bgcolor="rgba(255,255,255,0.95)",
                font=dict(size=12, color="#333333")
            ))
        elif year >= 2021:
            # Middle class squeeze
            middle_wage = texas_data[(texas_data['Wage Bin ID'] >= 7) & 
                                      (texas_data['Wage Bin ID'] <= 10)]['share'].sum() * 100
            annotations.append(dict(
                text=f"<b>Middle Wages ($50-100k):</b><br>{middle_wage:.1f}% of workers",
                xref="x", yref="y",
                x=texas_data['Wage Bin'].iloc[8],
                y=texas_data['share'].iloc[8] * 100 + 1.5,
                showarrow=True,
                arrowhead=2,
                arrowsize=1,
                arrowwidth=2,
                arrowcolor="#2A9D8F",
                ax=-60, ay=-30,
                bordercolor="#2A9D8F",
                borderwidth=2,
                borderpad=4,
                bgcolor="rgba(255,255,255,0.95)",
                font=dict(size=12, color="#333333")
            ))
        elif year <= 2014:
            # Earlier years - growth context
            annotations.append(dict(
                text=f"<b>Pre-Economic Boom Era</b><br>Stable wage distribution",
                xref="paper", yref="paper",
                x=0.5, y=0.5,
                showarrow=False,
                bgcolor="rgba(74,144,181,0.1)",
                bordercolor=color_us,
                borderwidth=2,
                borderpad=6,
                font=dict(size=13, color="#333333"),
                xanchor="center"
            ))
        
        return annotations

    for year in years:
        # Filter data for this year
        wage_texas_year = filtered_wage(wage_texas, year, 'Wage Bin ID')
        wage_us_year = filtered_wage(wage_us, year, 'Wage Bin ID')
        
        frame_data = [
            # Wage - Texas
            go.Bar(
                x=wage_texas_year['Wage Bin'],
                y=wage_texas_year['share'] * 100,
                name='Texas',
                marker_color=color_texas,
                hovertemplate='<b>%{x}</b><br>Texas: %{y:.2f}%<br><extra></extra>',
                legendgroup='region1',
                showlegend=True
            ),
            # Wage - US
            go.Bar(
                x=wage_us_year['Wage Bin'],
                y=wage_us_year['share'] * 100,
                name='United States',
                marker_color=color_us,
                hovertemplate='<b>%{x}</b><br>U.S.: %{y:.2f}%<br><extra></extra>',
                legendgroup='region2',
                showlegend=True
            )
        ]
        
        # Get dynamic annotations for this year
        frame_annotations = get_dynamic_wage_annotations(year, wage_texas_year, wage_us_year)
        
        frames.append(go.Frame(
            data=frame_data,
            name=str(year),
            layout=go.Layout(
                title_text=f'Wage Distribution Comparison ({year})<br>' +
                          '<sub>Comparing Texas with National Averages</sub>',
                annotations=frame_annotations
            )
        ))
    
    # Add initial data
    wage_texas_init = filtered_wage(wage_texas, initial_year, 'Wage Bin ID')
    wage_us_init = filtered_wage(wage_us, initial_year, 'Wage Bin ID')
    
    # Wage panel
    fig.add_trace(
        go.Bar(
            x=wage_texas_init['Wage Bin'],
            y=wage_texas_init['share'] * 100,
            name='Texas',
            marker_color=color_texas,
            hovertemplate='<b>%{x}</b><br>Texas: %{y:.2f}%<br><extra></extra>',
            legendgroup='region1'
        )
    )
    
    fig.add_trace(
        go.Bar(
            x=wage_us_init['Wage Bin'],
            y=wage_us_init['share'] * 100,
            name='United States',
            marker_color=color_us,
            hovertemplate='<b>%{x}</b><br>U.S.: %{y:.2f}%<br><extra></extra>',
            legendgroup='region2'
        )
    )
    
    # Add frames
    fig.frames = frames
    
    # Update axes - INCREASED FONT SIZES
    fig.update_xaxes(title_text="Wage Bracket", tickangle=-45, title_font=dict(size=16))
    fig.update_yaxes(title_text="Share of Workers (%)", range=[0, 18], title_font=dict(size=16))
    
    # Slider & buttons
    sliders = [dict(
        active=len(years) - 1,
        yanchor="top",
        y=-0.15,
        xanchor="left",
        x=0.1,
        currentvalue=dict(prefix="Year: ", visible=True, xanchor="center", font=dict(size=18)),
        pad=dict(b=10, t=10),
        len=0.8,
        transition=dict(duration=300),
        steps=[
            dict(
                args=[[str(year)], dict(frame=dict(duration=300, redraw=True), mode="immediate", transition=dict(duration=300))],
                method="animate",
                label=str(year)
            )
            for year in years
        ]
    )]
    
    # Layout - INCREASED FONT SIZES
    fig.update_layout(
        title={
            'text': f'Wage Distribution Comparison ({initial_year})<br><sub>Comparing Texas with National Averages</sub>',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 24}  # Increased from 20
        },
        barmode='group',
        height=1300,
        width=1500,
        showlegend=True,
        legend=dict(
            title=dict(text='<b>Region</b>', font=dict(size=16)),  # Increased from 12
            orientation='v',
            yanchor='top',
            y=0.98,
            xanchor='right',
            x=0.98,
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='gray',
            borderwidth=1,
            font=dict(size=14)  # Increased legend text size
        ),
        sliders=sliders,
        font=dict(size=13),  # Increased from 11
        hovermode='x unified',
        paper_bgcolor='white',
        plot_bgcolor='rgba(240,240,240,0.5)',
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                x=0.5,
                y=-0.25,
                xanchor="center",
                yanchor="top",
                pad=dict(r=10, t=10),
                buttons=[
                    dict(
                        label="Play",
                        method="animate",
                        args=[None, dict(
                            frame=dict(duration=500, redraw=True),
                            fromcurrent=True,
                            mode="immediate",
                            transition=dict(duration=300)
                        )]
                    ),
                    dict(
                        label="Pause",
                        method="animate",
                        args=[[None], dict(
                            frame=dict(duration=0, redraw=False),
                            mode="immediate",
                            transition=dict(duration=0)
                        )]
                    )
                ]
            )
        ]
    )
    
    # Updated annotation - Get initial annotations
    initial_annotations = get_dynamic_wage_annotations(initial_year, wage_texas_init, wage_us_init)
    
    # Add all initial annotations
    for annotation in initial_annotations:
        fig.add_annotation(annotation)
    
    return fig

# Create and show the interactive wage visualization with slider
fig_wage_interactive = create_interactive_wage_comparison_with_slider()
fig_wage_interactive.show()